In [ ]:
from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os


if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

In [ ]:
class CondEncoderWeighted(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=32, out_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU()
        )
        self.attn = nn.Linear(out_dim, 1)

    def forward(self, x):
        h = self.encoder(x)                      # [B, 4, out_dim]
        attn_scores = self.attn(h).squeeze(-1)   # [B, 4]
        attn_weights = F.softmax(attn_scores, dim=1).unsqueeze(-1)  # [B, 4, 1]
        return (h * attn_weights).sum(dim=1)     # [B, out_dim]


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.q = nn.Conv2d(in_channels, in_channels, 1)
        self.k = nn.Conv2d(in_channels, in_channels, 1)
        self.v = nn.Conv2d(in_channels, in_channels, 1)
        self.proj = nn.Conv2d(in_channels, in_channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q(x).reshape(B, C, -1)
        k = self.k(x).reshape(B, C, -1)
        v = self.v(x).reshape(B, C, -1)
        attn = torch.softmax(q.transpose(1,2) @ k / (C**0.5), dim=-1)
        out = (attn @ v.transpose(1,2)).transpose(1,2).reshape(B, C, H, W)
        return self.proj(out) + x

In [ ]:
class AdaIN(nn.Module):
    def __init__(self, channels, cond_dim):
        super().__init__()
        self.fc = nn.Linear(cond_dim, channels*2)
    def forward(self, x, cond):
        h = self.fc(cond)
        gamma, beta = h.chunk(2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)
        mean = x.mean([2,3], keepdim=True)
        std = x.std([2,3], keepdim=True)
        x_norm = (x - mean) / (std + 1e-5)
        return gamma * x_norm + beta


In [ ]:
def get_timestep_embedding(timesteps, embedding_dim):
    """
    timesteps: 1-D  (B,)  or 2-D (B,1) tensor of integers / floats
    returns:   (B, embedding_dim) sinusoidal embedding
    """
    if timesteps.ndim == 2:
        timesteps = timesteps.squeeze(-1)          # (B,)
    assert timesteps.ndim == 1                     # ensure 1-D
    half_dim = embedding_dim // 2
    exponents = torch.arange(half_dim, device=timesteps.device) / half_dim
    freqs = 10000 ** (-exponents)
    angles = timesteps.float()[:, None] * freqs[None, :]  # (B, half_dim)
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
    return emb                                    # (B, embedding_dim)


In [ ]:
class ImprovedResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, cond_dim=8, use_attention=False):
        super().__init__()
        self.same_channels = in_channels == out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.ada = AdaIN(out_channels, cond_dim)
        self.conv_proj = nn.Conv2d(in_channels, out_channels, 1) if not self.same_channels else nn.Identity()
        self.use_attention = use_attention
        if use_attention:
            self.attn = SelfAttention(out_channels)
        else:
            self.attn = nn.Identity()
    def forward(self, x, cond):
        h = F.gelu(self.norm1(self.conv1(x)))
        h = self.ada(h, cond)
        h = F.gelu(self.norm2(self.conv2(h)))
        h = self.ada(h, cond)  # <- Add this line
        h = self.attn(h)
        if self.same_channels:
            return (x + h) / 1.414
        else:
            x_proj = self.conv_proj(x)  # e.g., a 1x1 conv
            return (x_proj + h) / 1.414


In [ ]:
class CondSequential(nn.Module):
    """Sequential that passes (x, cond) to each sub-module."""
    def __init__(self, *layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, x, cond):
        for layer in self.layers:
            x = layer(x, cond)
        return x


In [ ]:
class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=1, base=128, cond_dim=128, time_dim=128):
        super().__init__()
        self.time_dim = time_dim
        self.time_embed = nn.Sequential(
            nn.Linear(time_dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, cond_dim)
        )
        self.cond_encoder = CondEncoderWeighted(in_dim=2, hidden_dim=32, out_dim=cond_dim)

        # ─── Encoder ───────────────────────────────────────────────
        self.enc1 = CondSequential(
            ImprovedResBlock(in_channels, base, cond_dim),
            ImprovedResBlock(base, base, cond_dim)
        )
        self.enc2 = CondSequential(
            ImprovedResBlock(base, base*2, cond_dim, use_attention=True),
            ImprovedResBlock(base*2, base*2, cond_dim, use_attention=True)
        )
        self.enc3 = CondSequential(
            ImprovedResBlock(base*2, base*4, cond_dim, use_attention=True),
            ImprovedResBlock(base*4, base*4, cond_dim, use_attention=True)
        )
        self.enc4 = CondSequential(
            ImprovedResBlock(base*4, base*8, cond_dim, use_attention=True),
            ImprovedResBlock(base*8, base*8, cond_dim, use_attention=True)
        )

        # ─── Bottleneck ────────────────────────────────────────────
        self.mid = CondSequential(
            ImprovedResBlock(base*8, base*8, cond_dim, use_attention=True),
            ImprovedResBlock(base*8, base*8, cond_dim, use_attention=True)
        )

        # ─── Decoder ───────────────────────────────────────────────
        self.up1 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.dec1 = CondSequential(
            ImprovedResBlock(base*8, base*4, cond_dim, use_attention=True),
            ImprovedResBlock(base*4, base*4, cond_dim, use_attention=True)
        )

        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.dec2 = CondSequential(
            ImprovedResBlock(base*4, base*2, cond_dim, use_attention=True),
            ImprovedResBlock(base*2, base*2, cond_dim, use_attention=True)
        )

        self.up3 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.dec3 = CondSequential(
            ImprovedResBlock(base*2, base, cond_dim),
            ImprovedResBlock(base, base, cond_dim)
        )

        self.out = nn.Conv2d(base, in_channels, 1)

    def forward(self, x, cond_raw, t, context_mask=0):
        pooled_cond = self.cond_encoder(cond_raw)
        if context_mask.dim() == 2:     # [B,1]
            context_mask = context_mask.unsqueeze(-1)  # -> [B,1,1]
        pooled_cond = pooled_cond * (1 - context_mask)  # broadcast to [B,4,2]

        t_emb = get_timestep_embedding(t, self.time_dim).to(x.device)
        pooled_cond = pooled_cond + self.time_embed(t_emb)


        # Encoder
        e1 = self.enc1(x, pooled_cond)
        e2 = self.enc2(F.avg_pool2d(e1, 2), pooled_cond)
        e3 = self.enc3(F.avg_pool2d(e2, 2), pooled_cond)
        e4 = self.enc4(F.avg_pool2d(e3, 2), pooled_cond)

        # Bottleneck
        h = self.mid(e4, pooled_cond)

        # Decoder
        d1 = self.up1(h)
        d1 = self.dec1(torch.cat([d1, e3], dim=1), pooled_cond)

        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e2], dim=1), pooled_cond)

        d3 = self.up3(d2)
        d3 = self.dec3(torch.cat([d3, e1], dim=1), pooled_cond)

        return self.out(d3)

In [ ]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [ ]:
class DDPM(nn.Module):
    """
    Denoising Diffusion Probabilistic Model w/ Classifier-Free Guidance.

    nn_model(x, c, t, context_mask) must accept:
        x : [B, 1, 32, 32]   current noised sample
        c : [B, 4, 2]        structured conditioning (mode, weight) pairs
        t : [B, 1]           normalized timestep (0..1)
        context_mask : [B, 1, 1]  1→drop conditioning, 0→use conditioning
                                  (will broadcast to [B, 4, 2] internally)
    """
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        super().__init__()
        self.nn_model  = nn_model.to(device)
        self.n_T       = n_T
        self.device    = device
        self.drop_prob = drop_prob
        self.loss_mse  = nn.MSELoss()

        # diffusion schedule buffers
        sched = ddpm_schedules(*betas, n_T)
        for k, v in sched.items():
            self.register_buffer(k, v)

    def forward(self, x, c):
        """
        Training loss (noise prediction).
        x : [B, 1, 32, 32] clean image
        c : [B, 4, 2]      structured conditioning
        """
        B = x.size(0)
        t_int = torch.randint(1, self.n_T + 1, (B,), device=self.device)  # [B]
        eps   = torch.randn_like(x)

        x_t = (self.sqrtab[t_int, None, None, None] * x +
               self.sqrtmab[t_int, None, None, None] * eps)

        # mask shape must broadcast over [B, 4, 2]; use [B,1,1]
        context_mask = torch.bernoulli(
            torch.full((B, 1, 1), self.drop_prob, device=self.device)
        )

        # normalize t to 0..1 and give as [B,1]
        t_norm = (t_int / self.n_T).unsqueeze(-1)  # [B,1]

        eps_pred = self.nn_model(x_t, c, t_norm, context_mask)
        return self.loss_mse(eps, eps_pred)

    @torch.no_grad()
    def sample(self, n_sample, size, device, c_i, guide_w=0.0):
        """
        Generate samples via reverse diffusion.

        c_i : [B, 4, 2] conditioning.
        guide_w:
            0   -> unconditional
            1   -> standard guidance
            >1  -> stronger guidance
        """
        x = torch.randn(n_sample, *size, device=device)  # start at x_T
        store = []

        for i in range(self.n_T, 0, -1):
            t_norm = torch.full((n_sample, 1), i / self.n_T, device=device)  # [B,1]

            # masks: broadcast over [B,4,2]
            mask_0 = torch.zeros(n_sample, 1, 1, device=device)  # keep cond
            mask_1 = torch.ones (n_sample, 1, 1, device=device)  # drop cond

            # predict noise with & without conditioning
            eps_cond   = self.nn_model(x, c_i,               t_norm, mask_0)
            eps_uncond = self.nn_model(x, torch.zeros_like(c_i), t_norm, mask_1)

            eps = eps_uncond + guide_w * (eps_cond - eps_uncond)

            z = torch.randn_like(x) if i > 1 else 0
            x = ( self.oneover_sqrta[i] *
                  (x - eps * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z )

            if i % 20 == 0 or i == self.n_T or i < 8:
                store.append(x.cpu().numpy())

        return x, np.array(store)


In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
from matplotlib.animation import FuncAnimation, PillowWriter

def train_waveguide(full_dataset):
    # ─── Hyper-params ────────────────────────────────────────────
    N_EPOCHS  = 50
    BATCH     = 256 if torch.cuda.is_available() else 64
    N_T       = 1_000
    DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
    COND_DIM  = 128   # new dimension *after* encoding [4,2] → [128]
    BASE      = 128
    LR        = 5e-5
    SAVE_DIR  = './data/diffusion_v4_4x2/'
    os.makedirs(SAVE_DIR, exist_ok=True)
    CKPT_PATH = os.path.join(SAVE_DIR, "ddpm_latest.pth")
    CURVE_PNG = os.path.join(SAVE_DIR, "loss_curve.png")

    # ─── Train / Test split (80 % / 20 %) ───────────────────────
    n_total   = len(full_dataset)
    n_train   = int(0.8 * n_total)
    n_val     = n_total - n_train
    train_ds, val_ds = random_split(full_dataset,
                                    [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  num_workers=8, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH,
                              shuffle=False, num_workers=8, pin_memory=True)

    # ─── Model & Optimiser ──────────────────────────────────────
    model = ImprovedUNet(
        in_channels=1,
        base=BASE,
        cond_dim=COND_DIM,   # internal encoded dimension
        time_dim=128
    ).to(DEVICE)

    ddpm  = DDPM(
        nn_model=model,
        betas=(1e-4, 0.03),
        n_T=N_T,
        device=DEVICE,
        drop_prob=0.1
    ).to(DEVICE)
    
    optim = torch.optim.Adam(ddpm.parameters(), lr=LR)

    train_losses, val_losses = [], []

    # ───────────────────── EPOCH LOOP ───────────────────────────
    for ep in range(N_EPOCHS):
        # ─── Train ──────────────────────────────────────────────
        ddpm.train()
        optim.param_groups[0]['lr'] = LR * (1 - ep / N_EPOCHS)
        running = 0.0
        tl_bar = tqdm(train_loader)
        for cond, params, imgs in tl_bar:
            imgs = imgs.to(DEVICE)              # [B, 1, 32, 32]
            cond = cond.to(DEVICE)              # [B, 4, 2]

            optim.zero_grad()
            loss = ddpm(imgs, cond)
            loss.backward()
            optim.step()
            running += loss.item() * imgs.size(0)
            tl_bar.set_description(f"Epoch {ep+1}/{N_EPOCHS}, loss={loss.item():.4f}")
        train_avg = running / len(train_loader.dataset)
        train_losses.append(train_avg)

        # ─── Validation ─────────────────────────────────────────
        ddpm.eval()
        running_val = 0.0
        with torch.no_grad():
            for cond, params, imgs in tqdm(val_loader):
                imgs = imgs.to(DEVICE)
                cond = cond.to(DEVICE)
                val_loss = ddpm(imgs, cond)
                running_val += val_loss.item() * imgs.size(0)
        val_avg = running_val / len(val_loader.dataset)
        val_losses.append(val_avg)

        print(f"Epoch {ep+1}: train {train_avg:.6f} | val {val_avg:.6f}")

        # ─── Save latest model (overwrite) ──────────────────────
        torch.save(ddpm.state_dict(), CKPT_PATH)

        # ─── Update & save loss curve ───────────────────────────
        plt.figure(figsize=(8,5))
        plt.plot(train_losses, label='train')
        plt.plot(val_losses,   label='val')
        plt.title("DDPM loss"); plt.xlabel("epoch"); plt.ylabel("MSE")
        plt.legend(); plt.grid()
        plt.tight_layout()
        plt.savefig(CURVE_PNG)
        plt.close()

    print(f"Finished. Final checkpoint at {CKPT_PATH}")

In [ ]:
from waveguide_dataset_paired import WaveguideDataset
dataset = WaveguideDataset('train_test_split.h5')

if __name__ == "__main__":
    train_waveguide(dataset)